In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import csv

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.62, phi0=0.4):
        self.phi0 = phi0
        self.entropic_unit = 136.6  # measured in MPa
        self.G = 0.13               # measured in MPa
        self.gamma = self.G/self.entropic_unit
        self.chi =  0.348
        self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm
        # self.delta = delta   # dimensionless
        
        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, 1.7)[0]

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, 1.9)[0]
        
        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit
            J= lambda1*lambda2*lambda3;
            phi = phi0/J;
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi))

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J))

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C = F.trans* F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit
        
        reference_energy_density =  self.reference_energy_density
        
        return 0.5*G*(Trace(C)) + nu*gel.H(J) - reference_energy_density


In [ ]:
### Main ###
L = 90.0
d = 1.62
w = 15.0
phi0 = 0.4
# order = 2
order =2
numIter = 15

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0)
mesh_file = 'meshes/mesh0.vol.gz'
mesh = Mesh(mesh_file)
# Draw(mesh)
fes = VectorH1(mesh, order=order, dirichlet="bottom")
print('nDoF = {}'.format(fes.ndof))

filename = f'gridfunctions/result_phi0=0.2_mesh0Polymer_order={order}_iter=' + str(numIter).zfill(2) 
u = GridFunction(fes)
u.Load(filename + '.gfu')
I = Id(mesh.dim)
F=I+Grad(u)
GF_energy_density = GridFunction(H1(mesh, order=1))
GF_energy_density.Set(gel.W(F))
Draw(GF_energy_density*1e3, mesh, deformation=u, min =0.0, max = 152.0)        # Energy densities in [KPa]
elasticEnergy = Integrate(GF_energy_density, mesh, order=5)
print('Total energy [mJ]: {:.2f}'.format(elasticEnergy))    

vtk = VTKOutput(mesh,coefs=[u, GF_energy_density],names=["u", "energy density"],filename=filename,subdivision=1)
vtk.Do()
    

nDoF = 586314


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: 249.44


'gridfunctions/result_phi0=0.2_mesh0Polymer_order=2_iter=15'